# AI Orchestrator — Colab Smoke Test

FastAPI 서버를 띄우지 않고 `pipeline.run()`을 **파이썬 함수로 직접 호출**해서
4가지 라우팅 경로를 그대로 검증하는 노트북이야.

**흐름:**
1. 레포 clone + 의존성 설치
2. HuggingFace 모델 ID 변수 설정 (router / summary / answer)
3. 각 모델 로드 & `LocalModelRegistry.register()`
4. `pipeline.run()`으로 시나리오 테스트
   - DIRECT_ANSWER
   - RETRIEVE_DOC (PDF 업로드)
   - SEARCH_PREP_THEN_RETRIEVE
   - ASK_CLARIFICATION


## 1. 레포 clone & 의존성 설치

In [ ]:
# 본인 레포 주소로 바꾸세요
REPO_URL = "https://github.com/YOUR_ORG/ai-orchestrator.git"
REPO_DIR = "/content/ai-orchestrator"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("repo already cloned")


In [ ]:
# 핵심 의존성 (이미 requirements.txt 있지만 Colab 기본 이미지 기준으로 추가 설치)
!pip install -q "fastapi==0.115.0" "pydantic==2.9.2" "pydantic-settings==2.5.2" \
    "pypdf==5.0.0" "sentence-transformers==3.1.1" "faiss-cpu==1.8.0" \
    "transformers>=4.45" "accelerate>=0.34" "bitsandbytes>=0.43" "nest_asyncio"


## 2. 환경변수 + sys.path 설정 **(반드시 app import 전에)**

In [ ]:
import os, sys

# Python 오케스트레이터가 local 백엔드를 쓰도록 전환
os.environ["LLM_BACKEND"] = "local"

# Colab GPU에 bge-m3(1024d) 올리기엔 부담되니 소형 임베딩으로 교체
os.environ["EMBEDDING_MODEL_NAME"] = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
os.environ["EMBEDDING_DIM"] = "384"
os.environ["EMBEDDING_DEVICE"] = "cuda"

# 업로드 파일 저장 경로 (Colab 임시)
os.environ["LOCAL_FILE_DIR"] = "/content/_orchestrator_files"

# 세션 캡 여유있게
os.environ["SESSION_MAX_BYTES"] = str(20 * 1024 * 1024)

# 레포를 import path에 추가
REPO_DIR = "/content/ai-orchestrator"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("env ready")


## 3. HuggingFace 모델 ID 설정

세 가지 role에 쓸 모델을 HF Hub ID로 지정해. 같은 모델을 여러 role에 공유해도 되고,
예산이 빠듯하면 `ROUTER_MODEL_ID`/`SUMMARY_MODEL_ID`를 `None`으로 두면
해당 role은 **휴리스틱/no-op 폴백**으로 동작해 (answer 모델만 있으면 파이프라인은 돈다).

In [ ]:
# 필요에 맞게 변경
ANSWER_MODEL_ID  = "Qwen/Qwen2.5-7B-Instruct"
ROUTER_MODEL_ID  = "Qwen/Qwen2.5-1.5B-Instruct"   # None 두면 휴리스틱 폴백
SUMMARY_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"     # None 두면 summary no-op

# 4bit 양자화 사용 (T4 16GB 기준 7B+3B+1.5B 동시 로드 가능)
USE_4BIT = True


## 4. 모델 로드 & 등록

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def _load(model_id: str):
    if model_id is None:
        return None, None
    print(f"loading {model_id} ...")
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    kwargs = dict(device_map="auto", trust_remote_code=True)
    if USE_4BIT:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
    else:
        kwargs["torch_dtype"] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    print(f"  -> ready on {next(model.parameters()).device}")
    return model, tok

answer_model,  answer_tok  = _load(ANSWER_MODEL_ID)
router_model,  router_tok  = _load(ROUTER_MODEL_ID)
summary_model, summary_tok = _load(SUMMARY_MODEL_ID)


In [ ]:
from app.services.llm.local_registry import LocalModelRegistry

LocalModelRegistry.clear()  # 노트북 재실행 안전

if answer_model is not None:
    LocalModelRegistry.register("answer", answer_model, answer_tok,
                                 device="cuda", max_new_tokens=512)

if router_model is not None:
    LocalModelRegistry.register("router", router_model, router_tok,
                                 device="cuda", max_new_tokens=24)

if summary_model is not None:
    LocalModelRegistry.register("summary", summary_model, summary_tok,
                                 device="cuda", max_new_tokens=400)

print("registered roles:", LocalModelRegistry.list_roles())


## 5. `pipeline.run()` 호출 헬퍼

Colab의 Jupyter 이벤트 루프 위에서 async 함수를 돌리려면 `nest_asyncio`가 필요해.

In [ ]:
import asyncio, nest_asyncio, uuid, json
nest_asyncio.apply()

# app 모듈 import (이 시점부터 config가 load됨 — 그래서 os.environ 설정을 먼저 해야 함)
from app.schemas.request import ChatRequest
from app.services.orchestrator.pipeline import run as pipeline_run
from app.services.session.session_manager import SessionManager
from app.services.embedding.embedding_singleton import EmbeddingSingleton

# 임베딩 모델 워밍업 (원래 FastAPI lifespan에서 하는 일)
EmbeddingSingleton.warmup()

def chat(session_id: str, user_text: str, file_path: str | None = None,
         file_name: str | None = None) -> dict:
    req = ChatRequest(
        session_id=session_id,
        user_text=user_text,
        file_path=file_path,
        file_name=file_name,
    )
    resp = asyncio.get_event_loop().run_until_complete(pipeline_run(req))
    return resp.model_dump()

def dump_session(session_id: str):
    s = asyncio.get_event_loop().run_until_complete(
        SessionManager.instance().get(session_id)
    )
    if s is None:
        print("(session purged or not found)")
        return
    print(json.dumps(s.model_dump(mode="json"), ensure_ascii=False, indent=2))

print("helper ready")


## 시나리오 A — `DIRECT_ANSWER`

PDF 없음, 평범한 질문. 라우터가 DIRECT_ANSWER를 반환하고 RAG는 안 탐.

In [ ]:
sid_a = f"test-a-{uuid.uuid4().hex[:6]}"

r1 = chat(sid_a, "Transformer 아키텍처가 뭐야?")
print("=== turn 1 ===")
print("answer_type:", r1["answer_type"])
print("answer     :", r1["answer"][:500])
print()

r2 = chat(sid_a, "그럼 attention은 그 안에서 어떤 역할이야?")
print("=== turn 2 (후속) ===")
print("answer_type:", r2["answer_type"])
print("answer     :", r2["answer"][:500])


## 시나리오 B — `RETRIEVE_DOC`

PDF 업로드 + 문서 키워드 포함 질문. 라우터가 RETRIEVE_DOC을 반환하고
ingest 완료를 기다린 뒤 FAISS retrieve → RAG 프롬프트로 answer LLM 호출.

In [ ]:
# 샘플 PDF 업로드 (또는 본인 PDF 경로 사용)
from google.colab import files
uploaded = files.upload()   # 파일 선택 창
pdf_name = list(uploaded.keys())[0]
pdf_path = f"/content/{pdf_name}"
with open(pdf_path, "wb") as f:
    f.write(uploaded[pdf_name])
print("saved to", pdf_path)


In [ ]:
sid_b = f"test-b-{uuid.uuid4().hex[:6]}"

r1 = chat(sid_b, "이 문서가 뭐에 대한 거야? 간단히 요약해줘.",
          file_path=pdf_path, file_name=pdf_name)
print("=== turn 1 ===")
print("answer_type:", r1["answer_type"])
print("answer     :", r1["answer"][:800])
print()

r2 = chat(sid_b, "이 문서에서 실험 결과 부분이 어떻게 나와 있어?")
print("=== turn 2 ===")
print("answer_type:", r2["answer_type"])
print("answer     :", r2["answer"][:800])


## 시나리오 C — `SEARCH_PREP_THEN_RETRIEVE`

PDF가 세션에 붙어있는 상태에서 참조 표현(`"그거"`, `"그 표"`)로 질문.
라우터가 SEARCH_PREP_THEN_RETRIEVE를 반환하고 search_prep으로 쿼리 재작성 후 retrieve.

In [ ]:
# 시나리오 B에서 쓴 session을 그대로 이어서 질문
r3 = chat(sid_b, "그거 다시 표로 정리해줘")
print("=== turn 3 (참조 표현) ===")
print("answer_type :", r3["answer_type"])
print("answer      :", r3["answer"][:800])

# 실제 라우터가 어느 label로 판정했는지 세션에서 확인
dump_session(sid_b)


## 시나리오 D — `ASK_CLARIFICATION`

빈 세션 + PDF 없음 + 지시어만 있는 짧은 질문. 라우터가 ASK_CLARIFICATION으로 판정하고
**LLM 호출 자체를 스킵**해서 고정 재질문 메시지를 반환.

In [ ]:
sid_d = f"test-d-{uuid.uuid4().hex[:6]}"

r = chat(sid_d, "그거 뭐야?")
print("answer_type:", r["answer_type"])
print("answer     :", r["answer"])
# LLM 호출이 없었으므로 GPU 부하 없이 즉시 반환되어야 함


## 세션 상태 덤프 (디버깅용)

Memory State Generator가 세션에 무엇을 남겼는지, router 결정이 무엇이었는지,
PDF ingest 상태가 어떤지 전부 JSON으로 볼 수 있어.

In [ ]:
dump_session(sid_a)
print("---")
dump_session(sid_b)


## 세션 정리

테스트 끝나면 메모리/파일/FAISS 모두 purge. (`SESSION_MAX_BYTES` 초과 시에도 자동으로 이게 일어남.)

In [ ]:
from app.services.session.session_cleaner import purge_session

for sid in [sid_a, sid_b, sid_d]:
    asyncio.get_event_loop().run_until_complete(purge_session(sid, reason="test_done"))
print("purged")
